# Contrôle global des données BAAC 2020-2024

Ce notebook a pour objectif de vérifier la cohérence globale des quatre jeux de données nettoyés avant leur intégration dans une base de données relationnelle.

Les contrôles portent notamment sur les dimensions des tables, leur granularité, leurs identifiants et les relations entre les différents fichiers.

In [35]:
# Import des bibliothèques

import pandas as pd

In [36]:
# 1.1. Chargement des fichiers nettoyés

df_caract = pd.read_csv(
    "data/processed/caract_2020_2024_clean.csv",
    sep=";",
    dtype={"Num_Acc": "string"}
)

df_lieux = pd.read_csv(
    "data/processed/lieux_2020_2024_clean.csv",
    sep=";",
    dtype={"Num_Acc": "string"}
)

df_vehicules = pd.read_csv(
    "data/processed/vehicules_2020_2024_clean.csv",
    sep=";",
    dtype={
        "Num_Acc": "string",
        "id_vehicule": "string"
    }
)

df_usagers = pd.read_csv(
    "data/processed/usagers_2020_2024_clean.csv",
    sep=";",
    dtype={
        "Num_Acc": "string",
        "id_usager": "string",
        "id_vehicule": "string"
    }
)

print("Caractéristiques :", df_caract.shape)
print("Lieux :", df_lieux.shape)
print("Véhicules :", df_vehicules.shape)
print("Usagers :", df_usagers.shape)

Caractéristiques : (268788, 16)
Lieux : (294438, 9)
Véhicules : (459137, 8)
Usagers : (612181, 11)


#### Résultat

Les quatre fichiers nettoyés ont été chargés correctement :

- Caractéristiques : 268 788 lignes et 16 variables
- Lieux : 294 438 lignes et 9 variables
- Véhicules : 459 137 lignes et 8 variables
- Usagers : 612 181 lignes et 11 variables

Les dimensions correspondent aux fichiers obtenus à la fin des étapes de préparation. Les quatre jeux de données sont donc disponibles pour les contrôles de cohérence globaux avant la modélisation relationnelle.

In [37]:
# 1.2. Vérification des identifiants

print("CARACTÉRISTIQUES")
print("Num_Acc manquants :", df_caract["Num_Acc"].isna().sum())
print("Num_Acc dupliqués :", df_caract["Num_Acc"].duplicated().sum())

print("\nVÉHICULES")
print("id_vehicule manquants :", df_vehicules["id_vehicule"].isna().sum())
print("id_vehicule dupliqués :", df_vehicules["id_vehicule"].duplicated().sum())

print("\nUSAGERS")
print("id_usager_projet manquants :", df_usagers["id_usager_projet"].isna().sum())
print("id_usager_projet dupliqués :", df_usagers["id_usager_projet"].duplicated().sum())

CARACTÉRISTIQUES
Num_Acc manquants : 0
Num_Acc dupliqués : 0

VÉHICULES
id_vehicule manquants : 0
id_vehicule dupliqués : 0

USAGERS
id_usager_projet manquants : 0
id_usager_projet dupliqués : 0


#### Résultat

Les identifiants retenus pour les tables principales sont complets et uniques :

- `Num_Acc` pour la table Caractéristiques ;
- `id_vehicule` pour la table Véhicules ;
- `id_usager_projet` pour la table Usagers.

Ces trois variables peuvent donc être utilisées comme clés primaires lors de la modélisation relationnelle.

La table Lieux n'est pas concernée par ce contrôle d'unicité sur `Num_Acc`, car un même accident peut être associé à plusieurs lignes de lieu.

In [38]:
# 1.3. Vérification des relations entre les tables

accidents = set(df_caract["Num_Acc"])
vehicules = set(df_vehicules["id_vehicule"])

print(
    "Lieux sans accident correspondant :",
    (~df_lieux["Num_Acc"].isin(accidents)).sum()
)

print(
    "Véhicules sans accident correspondant :",
    (~df_vehicules["Num_Acc"].isin(accidents)).sum()
)

print(
    "Usagers sans accident correspondant :",
    (~df_usagers["Num_Acc"].isin(accidents)).sum()
)

print(
    "Usagers sans véhicule correspondant :",
    (~df_usagers["id_vehicule"].isin(vehicules)).sum()
)

Lieux sans accident correspondant : 0
Véhicules sans accident correspondant : 0
Usagers sans accident correspondant : 0
Usagers sans véhicule correspondant : 0


#### Résultat

Aucune ligne orpheline n'a été détectée entre les quatre tables :

- tous les lieux correspondent à un accident existant ;
- tous les véhicules correspondent à un accident existant ;
- tous les usagers correspondent à un accident existant ;
- tous les usagers sont associés à un véhicule existant.

Les relations entre les quatre jeux de données sont donc cohérentes et pourront être utilisées pour la future modélisation relationnelle.

In [39]:
# 1.4. Vérification des cardinalités

print(
    "Maximum de lignes Lieux par accident :",
    df_lieux.groupby("Num_Acc").size().max()
)

print(
    "Maximum de véhicules par accident :",
    df_vehicules.groupby("Num_Acc").size().max()
)

print(
    "Maximum d'usagers par accident :",
    df_usagers.groupby("Num_Acc").size().max()
)

print(
    "Maximum d'usagers par véhicule :",
    df_usagers.groupby("id_vehicule").size().max()
)

Maximum de lignes Lieux par accident : 5
Maximum de véhicules par accident : 25
Maximum d'usagers par accident : 65
Maximum d'usagers par véhicule : 65


#### Résultat

Les cardinalités observées confirment que les tables présentent des relations de type un-à-plusieurs :

- un accident peut être associé à plusieurs lignes de lieu ;
- un accident peut impliquer plusieurs véhicules ;
- un accident peut concerner plusieurs usagers ;
- un véhicule peut être associé à plusieurs usagers.

Ces cardinalités sont cohérentes avec la structure des données BAAC et seront prises en compte dans la modélisation relationnelle.

In [40]:
# 1.5. Vérification de la couverture des accidents

print(
    "Accidents sans lieu :",
    (~df_caract["Num_Acc"].isin(df_lieux["Num_Acc"])).sum()
)

print(
    "Accidents sans véhicule :",
    (~df_caract["Num_Acc"].isin(df_vehicules["Num_Acc"])).sum()
)

print(
    "Accidents sans usager :",
    (~df_caract["Num_Acc"].isin(df_usagers["Num_Acc"])).sum()
)

Accidents sans lieu : 0
Accidents sans véhicule : 0
Accidents sans usager : 0


#### Résultat

Tous les accidents présents dans la table Caractéristiques sont représentés dans les trois autres tables :

- aucun accident sans information de lieu ;
- aucun accident sans véhicule ;
- aucun accident sans usager.

La couverture des 268 788 accidents est donc complète dans les quatre jeux de données.

## Conclusion

Le contrôle global des quatre jeux de données BAAC 2020-2024 confirme leur cohérence avant la modélisation relationnelle.

Les identifiants principaux sont complets et uniques, aucune ligne orpheline n'a été détectée et l'ensemble des 268 788 accidents est représenté dans les tables Lieux, Véhicules et Usagers.

Les cardinalités observées confirment les relations entre les différentes tables et permettent de préparer la création de la base de données relationnelle.

Les données nettoyées et validées sont désormais prêtes pour l'étape de modélisation et d'exploitation SQL.